# Implement logistic regression from scratch

## Simple gradient descent

In [1]:
import numpy as np

In [2]:
def sigmoid(input):
    return 1.0 / (1 + np.exp(-input))

In [3]:
def compute_prediction(X, weights):
    z = np.dot(X, weights)
    predictions = sigmoid(z)
    return predictions

In [4]:
def update_weights_gd(X_train, y_train, weights, learning_rate):
    predictions = compute_prediction(X_train, weights)
    weights_delta = np.dot(X_train.T, y_train - predictions)
    m = y_train.shape[0]
    weights += learning_rate / float(m) * weights_delta
    return weights

In [5]:
def compute_cost(X, y, weights):
    predictions = compute_prediction(X, weights)
    cost = np.mean(-y * np.log(predictions) - (1 - y) * np.log(1 - predictions))
    return cost

In [6]:
def train_logistic_regression(X_train, y_train, max_iter, learning_rate, fit_intercept=False):
    if fit_intercept:
        intercept = np.ones((X_train.shape[0], 1))
        X_train = np.hstack((intercept, X_train))
    weights = np.zeros(X_train.shape[1])
    for iteration in range(max_iter):
        weights = update_weights_gd(X_train, y_train, weights, learning_rate)
        if iteration % 100 == 0:
            print(compute_cost(X_train, y_train, weights))
    return weights

In [7]:
def predict(X, weights):
    if X.shape[1] == weights.shape[0] - 1:
        intercept = np.ones((X.shape[0], 1))
        X = np.hstack((intercept, X))
    return compute_prediction(X, weights)

In [8]:
X_train = np.array([[6, 7],
                    [2, 4],
                    [3, 6],
                    [4, 7],
                    [1, 6],
                    [5, 2],
                    [2, 0],
                    [6, 3],
                    [4, 1],
                    [7, 2]])

y_train = np.array([0,
                    0,
                    0,
                    0,
                    0,
                    1,
                    1,
                    1,
                    1,
                    1])

In [9]:
weights = train_logistic_regression(X_train, y_train, max_iter=1000, learning_rate=0.1, fit_intercept=True)

0.5744042371657099
0.034460223392510314
0.018265572708476157
0.012493458388041109
0.00951532913854744
0.007693388060646923
0.0064620943335143306
0.0055735118468322594
0.004901632254525041
0.004375567740665121


In [10]:
X_test = np.array([[6, 1],
                   [1, 3],
                   [3, 1],
                   [4, 5]])

In [11]:
predictions = predict(X_test, weights)

## Using it on ctr prediction

In [12]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
import timeit

In [13]:
n_rows = 300000
df = pd.read_csv("train.gz", nrows=n_rows)

In [14]:
X = df.drop(['click', 'id', 'hour', 'device_id', 'device_ip'], axis=1).values
Y = df['click'].values

In [15]:
n_train = 10000
X_train = X[:n_train]
Y_train = Y[:n_train]
X_test = X[n_train:]
Y_test = Y[n_train:]

In [16]:
enc = OneHotEncoder(handle_unknown='ignore')
X_train_enc = enc.fit_transform(X_train)
X_test_enc = enc.transform(X_test)

In [17]:
start_time = timeit.default_timer()
weights = train_logistic_regression(X_train_enc.toarray(), Y_train, max_iter=1000, learning_rate=0.01, fit_intercept=True)
print(f"--- {(timeit.default_timer() - start_time)}.3fs seconds ---")

0.6820019456743648
0.4608619713011896
0.4503715555130051
0.4455503890097847
0.4420611414384596
0.4393702812833892
0.437228041454526
0.4354781787758496
0.43401801289720104
0.4327779028622343
--- 16.230788496999594.3fs seconds ---


In [18]:
pred = predict(X_test_enc[:10000].toarray(), weights)
print(f'Training samples: {n_train}, AUC on testing set: {roc_auc_score(Y_test[:10000], pred):.3f}')

Training samples: 10000, AUC on testing set: 0.689


## Stochastic gradient descent

In [19]:
def update_weights_sgd(X_train, y_train, weights, learning_rate):
    for X_each, y_each in zip(X_train, y_train):
        prediction = compute_prediction(X_each, weights)
        weights_delta = X_each.T * (y_each - prediction)
        weights += learning_rate * weights_delta
    return weights

In [20]:
def train_logistic_regression_sgd(X_train, y_train, max_iter, learning_rate, fit_intercept=False):
    if fit_intercept:
        intercept = np.ones((X_train.shape[0], 1))
        X_train = np.hstack((intercept, X_train))
    weights = np.zeros(X_train.shape[1])
    for iteration in range(max_iter):
        weights = update_weights_sgd(X_train, y_train, weights, learning_rate)
        if iteration % 100 == 0:
            print(compute_cost(X_train, y_train, weights))
    return weights

In [21]:
start_time = timeit.default_timer()
weights = train_logistic_regression_sgd(X_train_enc.toarray(), Y_train, max_iter=1000, learning_rate=0.01,
                                        fit_intercept=True)
print(f"--- {(timeit.default_timer() - start_time)}.3fs seconds ---")

0.41496547913327814
0.35354754904027424
0.33959003103441565
0.3321135469202604
0.32738484092668396
0.3241190448939511
0.3217287654813196
0.31990311339146466
0.3184623109424547
0.31729549500473647
--- 72.65234925200639.3fs seconds ---


In [22]:
pred = predict(X_test_enc[:10000].toarray(), weights)
print(f'Training samples: {n_train}, AUC on testing set: {roc_auc_score(Y_test[:10000], pred):.3f}')

Training samples: 10000, AUC on testing set: 0.684
